# 🚀 OPERA POPE Benchmark on Kaggle (2× NVIDIA T4 GPUs)

This notebook runs the **POPE** (Prediction of Object Presence Evaluation) benchmark across all 3 splits (`adversarial`, `random`, `popular`) on **LLaVA-1.5-7B** or **Qwen2-VL-7B-Instruct** using the **OPERA** hallucination mitigation method.

### ⏱️ Latency & Efficiency Profiling (10 Samples / Split):
- Configured to run **10 samples per split** across **Adversarial, Random, and Popular** (total 30 samples) to accurately calculate the **average runtime per sample**.
- Uses `torch.cuda.synchronize()` for microsecond-precise GPU inference timing.
- Computes per-split latency and overall average execution time per sample.

### Key Technical Requirements & Configurations:
- **Hardware:** 2× NVIDIA T4 GPUs (32GB VRAM total) with `device_map="auto"`.
- **Precision:** BF16 Full Precision (`torch.bfloat16`) with automatic FP16 fallback. **No quantization** (no 4-bit / 8-bit bitsandbytes).
- **Decoding Strategy:** Strictly **Greedy Decoding** (`do_sample=False`, `temperature=0.0`, `max_new_tokens=6`).
- **Prompt Suffix:**
  - **LLaVA-1.5-7B:** Retains original question (no suffix).
  - **Qwen2-VL-7B:** Appends `" Please answer with yes or no." `.
- **Supported Models:** Both **LLaVA-1.5-7B** (`MODEL = "llava"`) and **Qwen2-VL-7B-Instruct** (`MODEL = "qwen2vl"`).
- **Splits:** Evaluates `--split all` (runs `random`, `popular`, `adversarial` sequentially with a single model load to maximize GPU efficiency).
- **Data Detection:** Auto-detects COCO val2014 images and POPE annotations from `/kaggle/input/`.

In [ ]:
# Cell 1: Environment Setup & Dependencies Installation
# 1. Gỡ bỏ torchaudio để giải quyết dứt điểm xung đột CUDA version mismatch
!pip uninstall -y -q torchaudio

# 2. Cài đặt các thư viện cần thiết (không cài torchvision/torch để tránh lệch CUDA Kaggle)
!pip install -q --no-cache-dir \
    "transformers>=4.45.0" \
    "accelerate>=0.26.0" \
    sentencepiece \
    protobuf \
    tiktoken \
    qwen_vl_utils \
    pyyaml \
    tqdm \
    huggingface_hub \
    pandas

print("✅ Dependencies successfully installed!")
from transformers import AutoProcessor
print("✅ AutoProcessor import verified successfully!")


### 🔐 Step 2: HuggingFace Authentication
Access HuggingFace Hub to download pretrained models (`llava-hf/llava-1.5-7b-hf` or `Qwen/Qwen2-VL-7B-Instruct`) using Kaggle Secrets (`HF_TOKEN`).

In [ ]:
# Cell 2: HuggingFace Login via Kaggle Secrets
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ HuggingFace login successful!")
except Exception as e:
    print(f"⚠️ Note on Kaggle Secret retrieval: {e}")
    print("If running outside Kaggle or without Kaggle Secrets, login manually via: huggingface-cli login")

### 📂 Step 3: Clone OPERA Repository
Clone the repository to `/kaggle/working` and change working directory to `opera_experiments`.

In [ ]:
# Cell 3: Clone OPERA repository and switch to experiments directory
import os
import sys

os.chdir("/kaggle/working")

# If repo not already present, clone it; if present, pull latest updates
if not os.path.exists("OPERA"):
    !git clone https://github.com/ntmy12/OPERA.git
    print("✅ OPERA repository cloned successfully!")
else:
    print("ℹ️ OPERA repository already exists. Pulling latest code...")
    !git -C OPERA pull

%cd /kaggle/working/OPERA/opera_experiments
print(f"Current Working Directory: {os.getcwd()}")

### 🖥️ Step 4: Hardware & COCO Dataset Verification
Verify that 2× NVIDIA T4 GPUs are active and that the COCO val2014 dataset is properly detected.

In [ ]:
# Cell 4: Hardware Verification & Dataset Auto-Detection
import os
import glob
import torch

print("=" * 70)
print("         HARDWARE & ENVIRONMENT VERIFICATION")
print("=" * 70)

# 1. GPU Check
gpu_count = torch.cuda.device_count()
print(f"🖥️ CUDA Available: {torch.cuda.is_available()}")
print(f"🖥️ Total GPUs Detected: {gpu_count}")

assert gpu_count >= 1, "❌ Error: No GPU detected. Please enable GPU accelerator in Kaggle Notebook Settings!"

for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    total_mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"   - GPU {i}: {name} ({total_mem:.2f} GB VRAM)")

if gpu_count >= 2:
    print("✅ 2× T4 GPUs available! device_map='auto' will shard across both cards.")
else:
    print(f"ℹ️ Running on {gpu_count} GPU.")

# Check BF16 capability
bf16_ok = torch.cuda.is_bf16_supported()
print(f"⚡ Native BF16 Supported: {bf16_ok}")
if not bf16_ok:
    print("ℹ️ T4 GPU architecture uses FP16 native cores; wrapper will auto-resolve optimal precision.")

# 2. COCO val2014 Dataset Validation
coco_candidates = [
    "/kaggle/input/datasets/biminhco/val2014/val2014",
    "/kaggle/input/val2014/val2014",
    "/kaggle/input/coco-2014-val/val2014",
    "/kaggle/input/coco-val2014/val2014",
    "/kaggle/input/coco2014/val2014",
]

coco_dir = None
for p in coco_candidates:
    if os.path.isdir(p):
        coco_dir = p
        break

if coco_dir is None:
    print("🔍 Scanning /kaggle/input/ for COCO_val2014_*.jpg...")
    for root, _, files in os.walk("/kaggle/input/"):
        for f in files[:50]:
            if f.startswith("COCO_val2014_") and f.lower().endswith((".jpg", ".jpeg", ".png")):
                coco_dir = root
                break
        if coco_dir:
            break

assert coco_dir is not None, "❌ Error: COCO val2014 image folder not found in /kaggle/input/! Please attach dataset."
image_count = len(glob.glob(os.path.join(coco_dir, "COCO_val2014_*.jpg")))
print(f"✅ COCO val2014 Image Directory: {coco_dir} ({image_count} images found)")
print("=" * 70)

### ⚡ Step 5: Execute POPE Benchmark on 10 Samples / Split (`--split all`)
Run benchmark across all 3 splits (`adversarial`, `random`, `popular`) on **10 samples per split** to calculate the **average runtime per sample**.
- Model is loaded **once** onto GPU(s) and evaluated consecutively across splits.
- Supports both models: set `MODEL = "llava"` (LLaVA-1.5-7B) or `MODEL = "qwen2vl"` (Qwen2-VL-7B-Instruct).
- Subsamples 10 samples per split via `--max_samples 10` for fast runtime profiling.

In [ ]:
# Cell 5: Run POPE Benchmark with OPERA (10 samples per split for latency testing)
# Choose model:
#   - 'llava'   : LLaVA-1.5-7B (llava-hf/llava-1.5-7b-hf)
#   - 'qwen2vl' : Qwen2-VL-7B-Instruct (Qwen/Qwen2-VL-7B-Instruct)
MODEL = "llava"  # 👈 Change to 'qwen2vl' if evaluating Qwen2-VL

# Number of samples per split (Adversarial, Random, Popular)
MAX_SAMPLES = 10  # 👈 Evaluates 10 samples per split to calculate average runtime per sample

print(f"🚀 Starting POPE Benchmark with OPERA")
print(f"   Model       : {MODEL.upper()}")
print(f"   Max Samples : {MAX_SAMPLES} per split (Total: {MAX_SAMPLES * 3} samples across 3 splits)")
print(f"   Splits      : Random, Popular, Adversarial")
print(f"   Decoding    : Greedy (max_new_tokens=6, do_sample=False)")

# Execute POPE benchmark for all 3 splits (10 samples each)
!python benchmarks/pope/run_pope.py \
    --model {MODEL} \
    --use_opera \
    --split all \
    --max_samples {MAX_SAMPLES} \
    --max_new_tokens 6 \
    --dtype bf16 \
    --device auto \
    --seed 42 \
    --coco_dir "{coco_dir}"

### 📊 Step 6: Results Consolidation & Average Runtime Calculation
Aggregate the metrics from all 3 splits (`random`, `popular`, `adversarial`) and report the **Average Runtime Per Sample** alongside standard POPE metrics (Accuracy, Precision, Recall, F1-Score).

In [ ]:
# Cell 6: Summarize, Calculate Average Runtime, and Display Results
import os
import glob
import json
import pandas as pd

# Find latest results folder for the selected model
result_dirs = sorted(glob.glob(f"results/{MODEL}_pope_opera_*"))

if not result_dirs:
    print("❌ No results found. Please check Cell 5 output for errors.")
else:
    latest_run = result_dirs[-1]
    print(f"📁 Results Directory: {latest_run}\n")

    summary_file = os.path.join(latest_run, "summary_metrics.json")
    if os.path.exists(summary_file):
        with open(summary_file, 'r', encoding='utf-8') as f:
            all_metrics = json.load(f)

        rows = []
        for split in ["random", "popular", "adversarial"]:
            if split in all_metrics:
                m = all_metrics[split]
                rows.append({
                    "Split": split.capitalize(),
                    "Samples": m.get("num_evaluated", m.get("Total", 0)),
                    "Accuracy (%)": m.get("Accuracy", 0.0),
                    "Precision (%)": m.get("Precision", 0.0),
                    "Recall (%)": m.get("Recall", 0.0),
                    "F1-Score (%)": m.get("F1", 0.0),
                    "Yes Ratio (%)": m.get("Yes_ratio", 0.0),
                    "Avg Time/Sample (s)": m.get("avg_time_per_sample_s", "N/A"),
                    "Total Time (s)": m.get("total_inference_time_s", "N/A"),
                })

        df = pd.DataFrame(rows)
        print("=" * 95)
        print(f"       POPE BENCHMARK RESULTS (10 SAMPLES/SPLIT) | Model: {MODEL.upper()} | Method: OPERA")
        print("=" * 95)
        display(df)
        print("=" * 95)

        # Detailed Timing Summary
        if "overall" in all_metrics:
            overall = all_metrics["overall"]
            tot_samples = overall.get("total_samples", 0)
            tot_time = overall.get("total_inference_time_s", 0.0)
            avg_time = overall.get("overall_avg_time_per_sample_s", 0.0)
            print("\n" + "⏱️ " + "=" * 65)
            print(f"   AVERAGE RUNTIME PER SAMPLE SUMMARY | {MODEL.upper()} + OPERA")
            print("=" * 68)
            print(f"   • Total Samples Evaluated : {tot_samples} (10 Random + 10 Popular + 10 Adversarial)")
            print(f"   • Total Inference Time    : {tot_time:.2f} seconds")
            print(f"   • AVERAGE TIME / SAMPLE   : {avg_time:.4f} seconds / sample")
            if avg_time > 0:
                print(f"   • Throughput              : {1.0 / avg_time:.2f} samples / second")
            print("=" * 68)
    else:
        print(f"⚠️ summary_metrics.json not found in {latest_run}")

    # Optional: Display comparison if both models have been run
    all_summary_files = sorted(glob.glob("results/*_pope_opera_*/summary_metrics.json"))
    if len(all_summary_files) > 1:
        comp_rows = []
        for s_file in all_summary_files:
            folder = os.path.basename(os.path.dirname(s_file))
            m_tag = folder.split("_pope_")[0]
            try:
                with open(s_file, 'r', encoding='utf-8') as f:
                    s_data = json.load(f)
                if "overall" in s_data:
                    ov = s_data["overall"]
                    comp_rows.append({
                        "Model": m_tag.upper(),
                        "Total Samples": ov.get("total_samples", 0),
                        "Total Time (s)": ov.get("total_inference_time_s", 0.0),
                        "Avg Time/Sample (s)": ov.get("overall_avg_time_per_sample_s", 0.0),
                        "Results Dir": folder
                    })
            except Exception:
                pass
        if comp_rows:
            print("\n📊 Cross-Model Runtime Comparison:")
            display(pd.DataFrame(comp_rows))
